In [1]:
!pip install pandas

In [2]:
import pandas as pd
import numpy as np

In [50]:
df = pd.read_parquet("./data/processed/train_split.parquet")

In [53]:
print(df.shape)

(2200000, 36)


In [55]:
print(f"Stockout rate: {df['stockout_flag'].mean()*100:.1f}%")

Stockout rate: 38.0%


In [57]:
# Comparing sales on stockout days v normal days
print("Average sales on normal days:  ", df[df['stockout_flag']==0]['sale_amount'].mean().round(3))
print("Average sales on stockout days:  ", df[df['stockout_flag']==1]['sale_amount'].mean().round(3))

Average sales on normal days:   1.002
Average sales on stockout days:   1.04


In [58]:
# For stock out days , replace sale_amount with rolling_mean_7  if its higher
# This estimates what demand would have been without the stockout
# For normal days, demand_corrected = sale_amount

df['demand_corrected'] = df['sale_amount']
stockout_mask = df['stockout_flag'] == 1

df.loc[stockout_mask,'demand_corrected'] = df.loc[stockout_mask].apply(
    lambda row: max(row['sale_amount'],row['rolling_mean_7']),
    axis=1
)

In [59]:
print("Before correction (stockout days):", df.loc[stockout_mask, 'sale_amount'].mean().round(3))
print("After correction (stockout days): ", df.loc[stockout_mask, 'demand_corrected'].mean().round(3))

Before correction (stockout days): 1.04
After correction (stockout days):  1.249


In [60]:
# Non-stockout days should be identical
normal_mask = df['stockout_flag'] == 0

unchanged = (df.loc[normal_mask, 'sale_amount'] == df.loc[normal_mask, 'demand_corrected']).all()
print(f"Non-stockout days unchanged: {unchanged}")

Non-stockout days unchanged: True


In [61]:
df.to_parquet("../data/processed/train_corrected.parquet", index=False)
print("Saved to data/processed/train_corrected.parquet")
print(f"Columns: {df.shape[1]}")

Saved to data/processed/train_corrected.parquet
Columns: 37
